In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_cv_model(path):
    model = models.swin_t(weights=None)
    model.head = nn.Linear(model.head.in_features, 3)
    state = torch.load(path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

model_fold1 = load_cv_model('trained_model/swin_cv_fold_1.pth')
model_fold2 = load_cv_model('trained_model/swin_cv_fold_2.pth')
model_fold3 = load_cv_model('trained_model/swin_cv_fold_3.pth')

template = pd.read_csv('submission.csv')
ids = template['id'].values

class TestDataset(Dataset):
    def __init__(self, ids, test_dir, transform=None):
        self.ids = ids
        self.test_dir = Path(test_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_path = self.test_dir / f"{self.ids[idx]}.jpg"
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = TestDataset(ids, 'test', transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

softmax = nn.Softmax(dim=1)
model_to_sub = {0: 1, 1: 2, 2: 0}
all_codes = []

with torch.no_grad():
    for imgs in tqdm(test_loader, desc="Ensemble 3-Fold CV"):
        imgs = imgs.to(device)
        prob1 = softmax(model_fold1(imgs))
        prob2 = softmax(model_fold2(imgs))
        prob3 = softmax(model_fold3(imgs))
        avg_prob = (prob1 + prob2 + prob3) / 3.0
        _, preds = torch.max(avg_prob, 1)
        all_codes.extend([model_to_sub[int(p)] for p in preds.cpu().numpy()])



Ensemble 3-Fold CV: 100%|██████████| 46/46 [00:17<00:00,  2.61it/s]


In [10]:
submission_df = pd.DataFrame({'id': ids, 'predicted': all_codes})
submission_df.to_csv('submission_ensemble_cv.csv', index=False)